# RSICD Adapter-CLIP — Session 2 of 5 — Adapter training (epochs 8-14)

Auto-generated from the rsicd-clip-adapter repo. Source of truth: `KAGGLE_RUNBOOK.md`.

**Before running this notebook:**
1. **Data (required)**: Click **+ Add data** in the right panel, search `thedevastator/rsicd-image-caption-dataset`, click **Add**. The dataset will appear at exactly `/kaggle/input/datasets/thedevastator/rsicd-image-caption-dataset/`.
2. **Previous-session Datasets** (sessions 2-5 only): click **+ Add data** → add any `rsicd-adapter-s*` or `rsicd-fullfinetune` Datasets you saved earlier
3. Settings: **Accelerator = GPU P100 or T4**, **Internet = ON**
4. Click **Save Version → Save Output** at the end of this session

Cell 2 will verify the dataset is at the exact expected path.

In [ ]:
# === Install + clone + env ===
# IMPORTANT: always cd to /kaggle/working (an absolute path) before
# cloning, so re-running this cell doesn't nest rsicd-clip-adapter
# inside itself.
%cd /kaggle/working
!rm -rf rsicd-clip-adapter
!pip install open_clip_torch faiss-cpu ftfy accelerate pyyaml -q
!git clone https://github.com/Vatsal057/rsicd-clip-adapter.git
%cd /kaggle/working/rsicd-clip-adapter
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
print("Repo cloned, deps installed.")
print(f"CWD: {os.getcwd()}")

In [ ]:
# === Sanity: GPU + dataset location ===
import torch, os
from pathlib import Path
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
print()

# The dataset is expected at this exact path (the user attached
# 'thedevastator/rsicd-image-caption-dataset' via + Add data).
EXPECTED = Path("/kaggle/input/datasets/thedevastator/rsicd-image-caption-dataset")
if not EXPECTED.exists():
    print(f"ERROR: dataset not found at {EXPECTED}")
    print()
    print("Fix:")
    print("  1. Right panel -> '+ Add data'")
    print("  2. Search 'thedevastator/rsicd-image-caption-dataset'")
    print("  3. Click 'Add'")
    print()
    print("Currently attached under /kaggle/input/:")
    try:
        for name in sorted(p.name for p in Path('/kaggle/input').iterdir()):
            print(f"   - {name}")
    except FileNotFoundError:
        print("   (no /kaggle/input/ directory)")
    DATA_OK = False
elif not (EXPECTED / 'train.csv').exists():
    print(f"ERROR: {EXPECTED} exists but does not contain train.csv.")
    print()
    print("Listing files at that path:")
    for p in sorted(EXPECTED.iterdir()):
        print(f"   - {p.name}")
    DATA_OK = False
else:
    os.environ['RSICD_ARCHIVE'] = str(EXPECTED)
    print(f"Using dataset: {EXPECTED}")
    csvs = sorted(p.name for p in EXPECTED.glob('*.csv'))
    print(f"  CSVs:   {csvs}")
    DATA_OK = True

## Required: attach the previous session's Dataset
Click **+ Add data** → search `rsicd-adapter-s1` → **Add**.

In [ ]:
# === Restore checkpoint from previous session ===
import shutil, os
from pathlib import Path
src = Path("/kaggle/input/rsicd-adapter-s1/adapter_best.pt")
if not src.exists():
    print(f"WARN: {src} not found. Did you forget to '+ Add data' rsicd-adapter-s1?")
else:
    Path("results/checkpoints").mkdir(parents=True, exist_ok=True)
    shutil.copy(src, "results/checkpoints/adapter_best.pt")
    print(f"Restored: {src}")
    # Also restore training history if it was saved
    src_h = Path("/kaggle/input/rsicd-adapter-s1/training_history_adapter.json")
    if src_h.exists():
        Path("results/metrics").mkdir(parents=True, exist_ok=True)
        shutil.copy(src_h, "results/metrics/training_history_adapter.json")
        print("Restored training history.")

In [ ]:
# === Train (target epoch 14) ===
import yaml, os
with open("configs/adapter_base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["training"]["num_epochs"] = 14
cfg["training"]["resume_from"] = "results/checkpoints/adapter_best.pt"

with open("configs/adapter_session.yaml", "w") as f:
    yaml.dump(cfg, f)
!python scripts/03_train_adapter.py configs/adapter_session.yaml adapter

In [ ]:
# === Package output for next session ===
# After this cell, click 'Save Version' (top right) with 'Save Output' enabled.
# Then go to the Output tab -> 'New Dataset' -> name it 'rsicd-adapter-s2'.
# The next session will attach this Dataset via '+ Add data'.
import shutil, os
out_dir = "/kaggle/working/rsicd-adapter-s2"
os.makedirs(out_dir, exist_ok=True)
src = "results/checkpoints/adapter_best.pt"
if os.path.exists(src):
    dst = os.path.join(out_dir, os.path.basename(src)) if not os.path.isdir(src) else out_dir
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(out_dir, os.path.basename(src)), dirs_exist_ok=True)
    else:
        shutil.copy(src, dst)
    print(f"  + {src} -> {out_dir}")
src = "results/metrics/training_history_adapter.json"
if os.path.exists(src):
    dst = os.path.join(out_dir, os.path.basename(src)) if not os.path.isdir(src) else out_dir
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(out_dir, os.path.basename(src)), dirs_exist_ok=True)
    else:
        shutil.copy(src, dst)
    print(f"  + {src} -> {out_dir}")
src = "results/metrics/adapter_results.json"
if os.path.exists(src):
    dst = os.path.join(out_dir, os.path.basename(src)) if not os.path.isdir(src) else out_dir
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(out_dir, os.path.basename(src)), dirs_exist_ok=True)
    else:
        shutil.copy(src, dst)
    print(f"  + {src} -> {out_dir}")
print(f"\nReady. Save this notebook version with 'Save Output' ON, then convert output to Dataset 'rsicd-adapter-s2'.")

## Done!
Save the version, then convert output to Dataset `rsicd-adapter-s2`.